# Week 4: Transfer Learning, BERT (Seminar)

### Using pretrained transformers (for fun, profit and 1 point)

There are many toolkits that let you access pretrained transformer models (like we used pretrained embeddings earlier), but the most powerful and convenient by far is 🤗[`huggingface/transformers`](https://github.com/huggingface/transformers). In this week's practice, you'll learn how to download, apply and modify pretrained transformers for a range of tasks. Buckle up, we're going in!


__Pipelines:__ if all you want is to apply a pretrained model, you can do that in one line of code using pipeline. Huggingface/transformers has a selection of pre-configured pipelines for masked language modelling, sentiment classification, question aswering, etc. ([see full list here](https://huggingface.co/transformers/main_classes/pipelines.html))

A typical pipeline includes:
* pre-processing, e.g. tokenization, subword segmentation
* a backbone model, e.g. bert finetuned for classification
* output post-processing

Let's see it in action:

In [1]:
import transformers

In [22]:
sentiment_clf = transformers.pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

sentiment_clf(["transformers library can be really useful!", "YSDA midterm is soon"])

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9959487915039062},
 {'label': 'NEGATIVE', 'score': 0.986364483833313}]

In [3]:
transformers.pipelines.SUPPORTED_TASKS.keys()

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'summarization', 'translation', 'text2text-generation', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-to-text', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'image-to-image', 'keypoint-matching'])

But how can we find out which model is suitable for chosen task in such a big models space?

Option 1: Using search and filters in [web](https://huggingface.co/models) (user-friendly)

Option 2: Using `huggingface_hub` library to access API from Python (if you want to automate some process)


In [2]:
import huggingface_hub

In [5]:
some_model = next(huggingface_hub.list_models())

some_model

ModelInfo(id='deepseek-ai/DeepSeek-OCR', author=None, sha=None, created_at=datetime.datetime(2025, 10, 17, 6, 22, 5, tzinfo=datetime.timezone.utc), last_modified=None, private=False, disabled=None, downloads=141221, downloads_all_time=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, likes=1369, library_name=None, tags=['safetensors', 'deepseek_vl_v2', 'deepseek', 'vision-language', 'ocr', 'custom_code', 'image-text-to-text', 'multilingual', 'arxiv:2510.18234', 'license:mit', 'region:us'], pipeline_tag='image-text-to-text', mask_token=None, card_data=None, widget_data=None, model_index=None, config=None, transformers_info=None, trending_score=1369, siblings=None, spaces=None, safetensors=None, security_repo_status=None, xet_enabled=None)

In [6]:
filter = (
    "sentiment-analysis",
    "pytorch",
    "ru",
)

filtered_models = huggingface_hub.list_models(
    filter=filter,
    sort="downloads",
    limit=10,
)

print(f"Filtered by {filter}:")
for model in filtered_models:
    print(f"- https://huggingface.co/{model.id} ({model.downloads} downloads, {model.likes} likes)")

Filtered by ('sentiment-analysis', 'pytorch', 'ru'):
- https://huggingface.co/yangheng/deberta-v3-base-absa-v1.1 (164075 downloads, 60 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-sentiment (142311 downloads, 29 likes)
- https://huggingface.co/r1char9/rubert-base-cased-russian-sentiment (6820 downloads, 12 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-ru-go-emotions (411 downloads, 9 likes)
- https://huggingface.co/yangheng/deberta-v3-large-absa-v1.1 (373 downloads, 20 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-cedr (262 downloads, 3 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-emotion-detection-ru-go-emotions (238 downloads, 4 likes)
- https://huggingface.co/seara/rubert-base-cased-russian-sentiment (195 downloads, 11 likes)
- https://huggingface.co/seara/rubert-tiny2-russian-emotion-detection-cedr (110 downloads, 1 likes)
- https://huggingface.co/oxygeneDev/sentiment-multilingual (5

In [7]:
filter = (
    "summarization",
    "pytorch",
    "en",
)

filtered_models = huggingface_hub.list_models(
    filter=filter,
    sort="downloads",
    limit=10,
)

print(f"Filtered by {filter}:")
for model in filtered_models:
    print(f"- https://huggingface.co/{model.id} ({model.downloads} downloads, {model.likes} likes)")

Filtered by ('summarization', 'pytorch', 'en'):
- https://huggingface.co/google-t5/t5-small (3795273 downloads, 498 likes)
- https://huggingface.co/facebook/bart-large-cnn (2733059 downloads, 1483 likes)
- https://huggingface.co/google-t5/t5-base (1415460 downloads, 754 likes)
- https://huggingface.co/google-t5/t5-3b (1312140 downloads, 48 likes)
- https://huggingface.co/sshleifer/distilbart-cnn-12-6 (877746 downloads, 297 likes)
- https://huggingface.co/google-t5/t5-large (252936 downloads, 223 likes)
- https://huggingface.co/philschmid/bart-large-cnn-samsum (172444 downloads, 264 likes)
- https://huggingface.co/google/pegasus-xsum (95757 downloads, 212 likes)
- https://huggingface.co/sshleifer/distilbart-xsum-12-6 (93304 downloads, 7 likes)
- https://huggingface.co/Falconsai/text_summarization (32457 downloads, 266 likes)


Imagine the situation when you have a long text to read and a lack of time. Luckily, you've got an option to use one of pipelines! But which one?...

**Task 1 (0.5 points)**
- Find a suitable pipeline and model for text below
- Apply model to long text to get a short one
- Pretty-print the result and give an opinion if short text is good or not



In [8]:
mlm_model = transformers.pipeline(
    task="summarization",
    model="t5-small"
)

Device set to use cuda:0


In [9]:
long_text = """
The widespread adoption of remote work, accelerated by global events in the early 2020s, has triggered a significant and likely permanent shift in how we think about the workplace. This transition away from the traditional central office is having profound and multifaceted effects on urban economies, reshaping everything from commercial real estate to local small businesses.

One of the most immediate and visible impacts has been on the commercial real estate sector. With companies downsizing their physical footprints or adopting fully remote models, demand for office space has plummeted. This has led to rising vacancy rates, downward pressure on commercial rent prices, and a re-evaluation of the financial viability of large office buildings. City governments, which often rely heavily on property taxes from these high-value commercial properties, are now facing substantial budget shortfalls.

Furthermore, the daily rhythm of city centers has changed dramatically. The decline in the number of commuters has had a ripple effect on local businesses that once thrived on their patronage. Lunchtime cafes, after-work bars, dry cleaners, and public transit systems have all experienced a significant drop in revenue. This "doughnut effect" describes a phenomenon where the economic activity hollows out in the city center and increases in suburban residential areas as people work from home and spend their money locally.

However, it's not all negative. This shift also presents new opportunities. Some urban planners see a chance to repurpose vacant office buildings into much-needed residential housing, which could help address housing shortages and revitalize neighborhoods by creating 24/7 communities. Additionally, the ability to work remotely has spurred a reversal of rural depopulation in some regions, as professionals seek a better quality of life outside of major metropolitan areas, potentially distributing economic growth more evenly.

In conclusion, the remote work revolution is fundamentally restructuring urban economies. While it presents serious challenges to established systems like commercial real estate and downtown commerce, it also opens the door to innovative urban renewal and a more geographically dispersed economic landscape. The long-term effects will depend on how effectively cities and businesses can adapt to this new, more flexible paradigm.
"""

short_text = mlm_model(long_text)[0]["summary_text"]

In [10]:
print(short_text)

the widespread adoption of remote work has triggered a significant and likely permanent shift in how we think about the workplace . the shift is reshaping everything from commercial real estate to local small businesses . this shift also presents new opportunities for urban planners to repurpose vacant office buildings .


Получилось!

In [11]:
assert len(long_text) / len(short_text) > 5, "Too long, didn't read"

One of possible semi-supervised tasks used while BERT training is Masked Language Modeling. So our model have some text prediction capabilities!



In [12]:
mlm_model = transformers.pipeline(
    task="fill-mask",
    model="bert-base-cased"
)

mlm_model("My name is [MASK] Shady!")

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


[{'score': 0.09115740656852722,
  'token': 22191,
  'token_str': 'Slim',
  'sequence': 'My name is Slim Shady!'},
 {'score': 0.025677746161818504,
  'token': 2791,
  'token_str': 'Captain',
  'sequence': 'My name is Captain Shady!'},
 {'score': 0.02322397567331791,
  'token': 3056,
  'token_str': 'Miss',
  'sequence': 'My name is Miss Shady!'},
 {'score': 0.016291731968522072,
  'token': 4479,
  'token_str': 'Jimmy',
  'sequence': 'My name is Jimmy Shady!'},
 {'score': 0.013304642401635647,
  'token': 13960,
  'token_str': 'Mister',
  'sequence': 'My name is Mister Shady!'}]

In order to make result more readable we can just take top-1 result:

In [13]:
mlm_model("My name is [MASK] Shady!")[0]["sequence"]

'My name is Slim Shady!'

**Task 2 (0.5 points)**
- Using BERT's ability to solve MLM task, find out answers on the following questions
- Perform some fact-checking, don't trust LLMs!

**Questions:**
- When YSDA was founded?
- Who invented radio first?
- What is the fifth Fibonacci number?

In [14]:
mlm_model = transformers.pipeline(
    task="fill-mask",
    model="bert-base-cased"
)
print(mlm_model("YSDA was founded in the year [MASK].")[0]["sequence"])
print(mlm_model("The radio was invented by [MASK].")[0]["sequence"])
print(mlm_model("The fifth Fibonacci number is equal to [MASK].")[0]["sequence"])

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


YSDA was founded in the year 2000.
The radio was invented by Fr.
The fifth Fibonacci number is equal to one.


Вышло сомнительно.

---

### The building blocks of a pipeline

Huggingface also allows you to access its pipelines on a lower level. There are two main abstractions for you:
* `Tokenizer` - converts from strings to token ids and back
* `Model` - a PyTorch `nn.Module` with pretrained weights

You can use such models as part of your regular PyTorch code: insert it as a layer in your model, apply to a batch of data, backpropagate, optimize, etc.

In [15]:
tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
model = transformers.AutoModel.from_pretrained("bert-base-uncased")

In [16]:
lines = [
    "Luke, I am your father.",
    "Life is what happens when you're busy making other plans.",
    "I have no idea what pneumonoultramicroscopicsilicovolcanoconiosis is."
]

tokens_info = tokenizer(lines, padding=True, truncation=True, return_tensors="pt")
print("Tokenized:")
print(tokens_info)

print("\nDetokenized:")
for i in range(3):
    print(tokenizer.decode(tokens_info['input_ids'][i]))

Tokenized:
{'input_ids': tensor([[  101,  5355,  1010,  1045,  2572,  2115,  2269,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  2166,  2003,  2054,  6433,  2043,  2017,  1005,  2128,  5697,
          2437,  2060,  3488,  1012,   102,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0],
        [  101,  1045,  2031,  2053,  2801,  2054,  1052,  2638,  2819, 17175,
         11314,  6444,  2594,  7352, 26461, 27572, 11261,  6767, 15472,  6761,
          8663, 10735,  2483,  2003,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0]]), 'attention_mask': tensor([[1, 1,

You can see some special tokens appeared besides our original text. They are usually used to give model some additional information, so model treats them in individual way.

You can list all special tokens used by tokenizer (moreover, you can add your own special tokens, but make sure you will show them to your model while training):

In [17]:
tokenizer._special_tokens_map

{'bos_token': None,
 'eos_token': None,
 'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]',
 'additional_special_tokens': []}

In [18]:
tokenizer("First sentence", "Second sentence", return_token_type_ids=True)

{'input_ids': [101, 2034, 6251, 102, 2117, 6251, 102], 'token_type_ids': [0, 0, 0, 0, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

It's ineffective to put all possible tokens in vocabulary, but one also want to handle all possible text sequences instead of putting UNK everywhere.

WordPiece tokenization is here to help!

In [19]:
reversed_vocab = {token_id: token for token, token_id in tokenizer.vocab.items()}

In [20]:
for token_id in tokens_info["input_ids"][2]:
    print(reversed_vocab[token_id.item()], end=' ')

[CLS] i have no idea what p ##ne ##um ##ono ##ult ##ram ##ic ##ros ##copic ##sil ##ico ##vo ##lc ##ano ##con ##ios ##is is . [SEP] 

Now you can apply tokenized data with model.

Depending on your task, you can use different part of output. For example, `[CLS]`-token output can be obtained by `pooler_output` key in model output.

In [21]:
import torch

In [22]:
with torch.no_grad():
    out = model(**tokens_info)

print(out['pooler_output'])

tensor([[-0.8854, -0.4722, -0.9392,  ..., -0.8081, -0.6955,  0.8748],
        [-0.9297, -0.5161, -0.9334,  ..., -0.9017, -0.7492,  0.9201],
        [-0.6808, -0.1979, -0.7096,  ..., -0.6691, -0.4557,  0.7595]])


Transformers knowledge hub: https://huggingface.co/transformers/



---



### Visualizing BERT

Interpretability of models is one of key factors of understanding their behaviour.

Neural Networks are harder to interpret than Classic ML models, but still it's not impossible!

Remember Attention mechanism? It's human-understandable concept: look closely to tokens which are more valuable for context of the current one.

In [49]:
!pip install bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 8.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 6.4 MB/s eta 0:00:00


In [50]:
from transformers import AutoTokenizer, AutoModel, utils
from bertviz import model_view, head_view

input_text = "Every time I try to interpret BERT model behaviour, I find new interesting patterns"
model = AutoModel.from_pretrained("bert-base-cased", output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

inputs = tokenizer.encode(input_text, return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


<IPython.core.display.Javascript object>

In [51]:
head_view(attention, tokens)

<IPython.core.display.Javascript object>

Another possible task for BERT training is Next Sentence Prediction.

How BERT's heads looks at tokens in that case?

In [52]:
inputs = tokenizer.encode("I'm waiting for important call", "I can't go out right now", return_tensors="pt")
outputs = model(inputs)
attention = outputs[-1]
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

model_view(attention, tokens)

<IPython.core.display.Javascript object>

In [53]:
head_view(attention, tokens)

<IPython.core.display.Javascript object>

It looks interesting, doesn't it?

If you want to find out more about attention patterns, you can refer to special "field" of science - [BERTology](https://huggingface.co/docs/transformers/main/en/bertology).



---



### Tuning pretrained transfomers (for your own task and 2 points)

Important benefit of using big models is their ability to adapt to various tasks without spending a lot of time and resources for full training.

You could've heard about backbone models in another ML tasks, when they're tuned using specific data.

It's possible to tune model's weights directly, but you also can freeze model, use its outputs as knowledge and then extract neccessary information using much smaller neural networks.

#### Introduction

Here's an example of tuned BERT base model for Named Entity Recognition (NER) task:

In [3]:
tokenizer = transformers.AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = transformers.AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [4]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

As you can see, there's an additional classifier besides original BERT content. That layer is used to predict NER-classes for each BERT's token output.

BERT is suitable for tuning for different tasks since it outputs token embeddings and the whole data embedding in `[CLS]`-token as well.

#### Data preparation

In [18]:
import datasets

In [19]:
dataset = datasets.load_dataset("lhoestq/conll2003")

In [20]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [21]:
dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

Since BERT tokenization is different from the dataset's one, we need to fix that divergence.

**Task 3 (0.5 points)**
- Align dataset token labels to WordPiece tokens
- Handle special tokens as well

In [22]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(samples):
    tokenized_inputs = tokenizer(samples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    labels.append(0)
    for i, original_labels in enumerate(samples["ner_tags"]):
        num_tkn = len(tokenizer(samples["tokens"][i]).input_ids)-2
        labels.extend([original_labels]*num_tkn)
        
    labels.append(0)
    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [23]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

In [24]:
tokenized_dataset['train'][1]

{'id': '1',
 'tokens': ['Peter', 'Blackburn'],
 'pos_tags': [22, 22],
 'chunk_tags': [11, 12],
 'ner_tags': [1, 2],
 'input_ids': [101, 1943, 14428, 102],
 'token_type_ids': [0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1],
 'labels': [0, 1, 2, 0]}

In [25]:
tokenized_dataset["train"][2]

{'id': '2',
 'tokens': ['BRUSSELS', '1996-08-22'],
 'pos_tags': [22, 11],
 'chunk_tags': [11, 12],
 'ner_tags': [5, 0],
 'input_ids': [101,
  26660,
  13329,
  12649,
  15928,
  1820,
  118,
  4775,
  118,
  1659,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [0, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0]}

In [27]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Now dataset is ready to be used by BERT.

#### Model preparation

For our task we can use `AutoModelForTokenClassification`, which already provides required architecture with token classifier (e.g. classifier itself, class outputs).

You can handle these things by yourself: create PyTorch model class, init BERT model and Linear layer for classification, then override forward method and so on...

`AutoModelForTokenClassification` is chosen for the sake of simplicity, but it's still required for MLE to be capable of doing it with bare hands.

In [28]:
from transformers import AutoModelForTokenClassification

id2label = {0: "O", 1: "B-PER", 2: "I-PER", 3: "B-ORG", 4: "I-ORG", 5: "B-LOC", 6: "I-LOC", 7: "B-MISC", 8: "I-MISC"}
label2id = {label: id for id, label in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=9,
    id2label=id2label,
    label2id=label2id,
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Evaluation

Evaluation is crucial while writing papers or reporting your work results. Sometimes it can be tricky and own implementation can be buggy, so it usually preferred to calculate metrics using frameworks.

In [55]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 287.5 kB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=ba8020660753ada0053f0f1ba5e5f28990dfdd7eb4f5592e5b98468c1344f691
  Stored in directory: /home/eliyashev/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


Let's prepare `compute_metrics` function for the following training loop:

In [29]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score
from seqeval.scheme import IOB2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    y_true = []
    y_pred = []
    for i in range(len(predictions)):
        y_true_sample = []
        y_pred_sample = []
        for j in range(len(predictions[i])):
            if labels[i][j] == -100:
                continue

            y_true_sample.append(id2label[int(labels[i][j])])
            y_pred_sample.append(id2label[int(predictions[i][j])])

        y_true.append(y_true_sample)
        y_pred.append(y_pred_sample)

    return {
        "precision": precision_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "recall": recall_score(y_true, y_pred, mode="strict", scheme=IOB2),
        "f1": f1_score(y_true, y_pred, mode="strict", scheme=IOB2),
    }

#### Training

**Task 4 (0.5 points)**
- Choose proper hyperparameters for tuning the model
- Setup HF Trainer
- Check correctness using training results


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-ner",
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=50,
    logging_dir="./logs",
    report_to="none",
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
)

In [58]:
trainer = Trainer(model = model,
    args = training_args,
    data_collator = data_collator,train_dataset = tokenized_dataset["train"],eval_dataset = tokenized_dataset["validation"],
    compute_metrics = compute_metrics,
)

/usr/bin/ld: невозможно найти -laio: Нет такого файла или каталога
collect2: error: ld returned 1 exit status


MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)

Let's check metrics before training:

In [ ]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate(tokenized_dataset["test"])
print(results)

Compare test metrics before and after training. Did we succeed?

**Task 5 (1 point)**
- Compare our model's result with `dslim/bert-base-NER`
- Try to improve our model's quality. Choose any option:
  - Play with training hyperparameters (batch_size, lr, epochs, etc.)
  - Apply some training techniques (warm-up, lr-scheduling, etc.)
  - Perform error analysis and find model's weak spots (this option doesn't require fixing them)
  - Your very own idea
- Write a small report (up to 5 steps, results and conclusions) on the work done in "Tuning pretrained transformers" part

In [6]:
tokenizer = transformers.AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = transformers.AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

ner_pipeline_bert = transformers.pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [18]:
from transformers import Trainer, TrainingArguments

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
)

predictions = trainer.predict(test["validation"])

/tmp/ipykernel_8108/1775720622.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/bin/ld: невозможно найти -laio: Нет такого файла или каталога
collect2: error: ld returned 1 exit status


MissingCUDAException: CUDA_HOME does not exist, unable to compile CUDA op(s)

In [ ]:
ner_pipeline_bert(test['tokens'])

In [17]:
ner_pipeline_bert(test['tokens'])

[[],
 [],
 [{'entity': 'B-ORG',
   'score': 0.7770893,
   'index': 1,
   'word': 'J',
   'start': 0,
   'end': 1},
  {'entity': 'I-LOC',
   'score': 0.6719664,
   'index': 2,
   'word': '##AP',
   'start': 1,
   'end': 3},
  {'entity': 'I-ORG',
   'score': 0.46225473,
   'index': 3,
   'word': '##AN',
   'start': 3,
   'end': 5}],
 [],
 [{'entity': 'B-ORG',
   'score': 0.5639978,
   'index': 1,
   'word': 'L',
   'start': 0,
   'end': 1},
  {'entity': 'B-LOC',
   'score': 0.48483726,
   'index': 2,
   'word': '##UC',
   'start': 1,
   'end': 3},
  {'entity': 'I-ORG',
   'score': 0.6899351,
   'index': 3,
   'word': '##K',
   'start': 3,
   'end': 4},
  {'entity': 'I-LOC',
   'score': 0.5368309,
   'index': 4,
   'word': '##Y',
   'start': 4,
   'end': 5}],
 [],
 [],
 [{'entity': 'B-ORG',
   'score': 0.99643457,
   'index': 1,
   'word': 'CH',
   'start': 0,
   'end': 2},
  {'entity': 'I-ORG',
   'score': 0.8533952,
   'index': 2,
   'word': '##IN',
   'start': 2,
   'end': 4},
  {'enti

In [13]:
test = dataset["test"][0]

In [16]:
test['tokens']

['SOCCER',
 '-',
 'JAPAN',
 'GET',
 'LUCKY',
 'WIN',
 ',',
 'CHINA',
 'IN',
 'SURPRISE',
 'DEFEAT',
 '.']

In [ ]:
ner_pipeline_bert.